# One-head nanoGPT — Muon + auxiliary AdamW baseline

This notebook runs the **same one-block, one-attention-head nanoGPT** on a pinned, document-disjoint **FineWeb-Edu** corpus. It does not use Tiny Shakespeare.

The full reference uses three independent seeds. Checkpoints are restartable. The training device is Apple **MPS** when available, with CPU fallback. WeightWatcher runs at epoch zero and every nominal epoch using exactly:

```python
watcher.analyze(ERG=True, randomize=True, plot=False, min_evals=20)
```

The notebook plots `train_accuracy`, `test_accuracy`, `train_loss`, `test_loss`, `test_perplexity`, fixed-continuation `test_bleu`, matrix `alpha`, `ERG_gap`, and direct WeightWatcher `num_traps`. Curves show individual seeds plus the run-level mean and **95% Student-t** confidence interval.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd / "baseline" / "nanogpt_one_head"]
EXPERIMENT_ROOT = next(
    (path for path in candidates if (path / "configs" / "reference.yaml").is_file()),
    None,
)
if EXPERIMENT_ROOT is None:
    raise FileNotFoundError("Run from baseline/nanogpt_one_head or the repository root")
sys.path.insert(0, str(EXPERIMENT_ROOT / "src"))

from rg_nanogpt_one_head import (
    MATRIX_COLORS,
    canonical_seeds,
    choose_device,
    load_config,
    load_epoch_metrics,
    load_layer_metrics,
    plot_epoch_metric,
    plot_layer_metric,
    roots,
    run_optimizer_replicates,
    run_status_table,
)

CONFIG_PATH = EXPERIMENT_ROOT / "configs" / "reference.yaml"
CONFIG = load_config(CONFIG_PATH)
SEEDS = canonical_seeds(CONFIG)
PATHS = roots()
DEVICE = choose_device("auto")
PATHS["results"].mkdir(parents=True, exist_ok=True)
PATHS["plots"].mkdir(parents=True, exist_ok=True)
print("experiment root:", EXPERIMENT_ROOT)
print("device:", DEVICE)
print("data root:", PATHS["data"])
print("results root:", PATHS["results"])
print("seeds:", SEEDS)
display(pd.DataFrame([CONFIG["model"]]))
display(pd.DataFrame(CONFIG["optimizer_profiles"]).T)
display(pd.DataFrame({"matrix_type": list(MATRIX_COLORS), "color": list(MATRIX_COLORS.values())}))

## Run or resume the three replicates

The data preparation is idempotent. The first execution downloads the pinned FineWeb-Edu stream and writes exact 10M/1M/1M-token train/validation/test splits. Existing complete runs are skipped; incomplete runs resume from `checkpoint_latest.pt`.

In [ ]:
OPTIMIZER = "muon"
run_dirs = run_optimizer_replicates(
    cfg=CONFIG,
    config_path=CONFIG_PATH,
    optimizer_name=OPTIMIZER,
    seeds=SEEDS,
    data_root=PATHS["data"],
    results_root=PATHS["results"],
    device=str(DEVICE),
    resume=True,
    overwrite=False,
    prepare_data=True,
    progress=True,
)
print("run directories:")
for path in run_dirs:
    print(" -", path)
display(run_status_table(PATHS["results"], optimizers=[OPTIMIZER], seeds=SEEDS))

## Per-epoch task metrics

The test split stays held out during optimization. After training, the final and validation-selected checkpoints receive the test and deterministic greedy-continuation BLEU audits; these never select checkpoints, change learning rates, or tune hyperparameters. BLEU is a secondary language-model diagnostic, not a translation benchmark.

In [ ]:
epoch_metrics = load_epoch_metrics(
    PATHS["results"], optimizers=[OPTIMIZER], seeds=SEEDS
)
display(epoch_metrics.sort_values(["seed", "nominal_epoch"]))

performance_metrics = [
    "train_loss", "val_loss", "test_loss",
    "train_accuracy", "val_accuracy", "test_accuracy",
    "train_perplexity", "val_perplexity", "test_perplexity",
    "test_bleu", "primary_lr", "auxiliary_lr",
]
plot_dir = PATHS["plots"] / OPTIMIZER
for metric in performance_metrics:
    if metric not in epoch_metrics or epoch_metrics[metric].notna().sum() == 0:
        continue
    plot_epoch_metric(
        epoch_metrics,
        metric=metric,
        optimizers=[OPTIMIZER],
        title=f"{OPTIMIZER}: {metric} by epoch (95% Student-t CI)",
        output=plot_dir / f"{metric}.png",
    )
    plt.show()

## Per-matrix WeightWatcher trajectories

The same colors identify `W_Q`, `W_K`, `W_V`, `W_O`, `W_MLP_IN`, and `W_MLP_OUT` in every optimizer notebook. `alpha`, `ERG_gap`, and `num_traps` are direct WeightWatcher outputs; no fallback alpha or proxy trap count is used.

In [ ]:
layer_metrics = load_layer_metrics(
    PATHS["results"], optimizers=[OPTIMIZER], seeds=SEEDS
)
display(
    layer_metrics[
        ["optimizer", "seed", "epoch", "matrix_type", "alpha", "ERG_gap", "num_traps"]
    ].sort_values(["seed", "epoch", "matrix_type"])
)
for metric in ["alpha", "ERG_gap", "num_traps"]:
    plot_layer_metric(
        layer_metrics,
        optimizer=OPTIMIZER,
        metric=metric,
        title=f"{OPTIMIZER}: layer {metric} (95% Student-t CI)",
        output=plot_dir / f"layer_{metric}.png",
    )
    plt.show()

## Persisted artifacts

Each seed directory contains `manifest.json`, `metrics.csv`, `epoch_metrics.csv`, raw and aggregate WeightWatcher CSV files, model-only epoch checkpoints, `checkpoint_latest.pt` for restart, `checkpoint_best.pt`, `checkpoint_final.pt`, `test_results.json`, and `run_complete.json`.